# DatasetFilterResolver Tutorial

This tutorial demonstrates how to use the `DatasetFilterResolver` to filter samples across heterogeneous datasets using external YAML configuration.

## Overview

The `DatasetFilterResolver` solves a common problem when working with multiple genomics datasets: **experimental conditions are structured differently across datasets**. Some datasets define conditions at the repository level (all samples share the same conditions), while others define conditions per-sample in field definitions.

Instead of writing dataset-specific code, you specify:

1. **Filter criteria**: Which values you want (e.g., `carbon_source: ["D-glucose", "D-galactose"]`)
2. **Property mappings**: Where to find each property in each dataset (e.g., repo-level vs field-level)

## Key Features

- **Two-level filtering**:
  - **Level 1** (Repo/Config): Excludes entire datasets that don't match
  - **Level 2** (Field): Returns specific field values that match within datasets
- **Three output modes**:
  - `conditions`: Just matching field values (no data retrieval)
  - `samples`: Sample-level metadata (one row per sample_id)
  - `full_data`: All measurements for matching samples
- **External configuration**: No hardcoded dataset logic in your analysis code
- **Automatic path resolution**: Handles `experimental_conditions` prepending automatically

## Setup

In [1]:
from pathlib import Path
import yaml
import pandas as pd
from tfbpapi.datainfo.filter_resolver import DatasetFilterResolver
from tfbpapi.datainfo import DataCard

/home/chase/code/tfbp/tfbpapi/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Understanding Dataset Structures

Before filtering, let's examine how experimental conditions are structured in two representative datasets.

### Repo-Level Conditions: Kemmeren 2014

All samples in this dataset share the same experimental conditions defined at the repository level.

In [2]:
# Load Kemmeren 2014 datacard
kemmeren_card = DataCard("BrentLab/kemmeren_2014")

# Get repo-level experimental conditions
exp_conds = kemmeren_card.get_experimental_conditions("kemmeren_2014")

print("Kemmeren 2014 - Repo-Level Experimental Conditions")
print("=" * 60)
print(f"Temperature: {exp_conds.get('temperature_celsius')}°C")
print(f"Media: {exp_conds.get('media', {}).get('name')}")

carbon_source = exp_conds.get('media', {}).get('carbon_source', [])
if carbon_source:
    compounds = [cs.get('compound') for cs in carbon_source]
    print(f"Carbon source: {compounds}")

print("\nAll samples in this dataset share these conditions.")

Kemmeren 2014 - Repo-Level Experimental Conditions
Temperature: 30°C
Media: synthetic_complete
Carbon source: ['D-glucose']

All samples in this dataset share these conditions.


### Field-Level Conditions: Harbison 2004

This dataset has different experimental conditions for different samples, defined in the `condition` field's definitions.

In [3]:
# Load Harbison 2004 datacard
harbison_card = DataCard("BrentLab/harbison_2004")

# Check for repo-level conditions
exp_conds = harbison_card.get_experimental_conditions("harbison_2004")
print(f"Repo-level experimental_conditions: {exp_conds}")

# Get field-level definitions
condition_defs = harbison_card.get_field_definitions("harbison_2004", "condition")

print("\nHarbison 2004 - Field-Level Definitions")
print("=" * 60)
print(f"Number of condition definitions: {len(condition_defs)}")
print(f"Condition names: {list(condition_defs.keys())}")

# Show examples
print("\nExample conditions:")
for cond_name in ["YPD", "GAL", "HEAT"]:
    if cond_name in condition_defs:
        definition = condition_defs[cond_name]
        media = definition.get('media', {})
        carbon = media.get('carbon_source', [])
        compounds = [cs.get('compound') for cs in carbon] if carbon else []
        temp = definition.get('temperature_celsius', 'varies')
        print(f"  {cond_name}: {compounds}, {temp}°C")

print("\nEach sample can have a different condition value.")

Repo-level experimental_conditions: {}

Harbison 2004 - Field-Level Definitions
Number of condition definitions: 14
Condition names: ['YPD', 'SM', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90', 'Thi-', 'GAL', 'HEAT', 'Pi-', 'RAFF']

Example conditions:
  YPD: ['D-glucose'], 30°C
  GAL: ['D-galactose'], 30°C
  HEAT: ['D-glucose'], varies°C

Each sample can have a different condition value.


## Configuration Format

The filter configuration has two main sections:

```yaml
# 1. Filter criteria - what values you want
filters:
  carbon_source: ["D-glucose", "D-galactose"]
  temperature_celsius: [30]

# 2. Property mappings - where to find each property in each dataset
BrentLab/repo_name:
  property1:
    path: media.name  # Repo-wide property
  
  dataset:
    dataset_name:
      property2:
        path: temperature_celsius  # Dataset-specific property
      
      property3:
        field: condition  # Field-level property
        path: media.carbon_source
```

### Three Types of Property Configurations

1. **Repo-wide**: Property applies to ALL datasets in the repository
   - Paths automatically get `experimental_conditions.` prepended
   - Example: `path: media.name` → resolves to `experimental_conditions.media.name`

2. **Dataset-specific**: Property applies only to a specific dataset (at config level)
   - Also gets `experimental_conditions.` prepended
   - Example: `path: temperature_celsius` → resolves to `experimental_conditions.temperature_celsius`

3. **Field-level**: Property varies per sample, defined in field definitions
   - Requires `field` parameter to specify which field
   - Path is used directly on field definitions (NO prepending)
   - Example: `field: condition, path: media.carbon_source` → looks in condition field definitions

## Example 1: Basic Filtering by Carbon Source

Let's filter for samples grown on D-glucose in Harbison 2004.

In [4]:
# Create filter configuration
glucose_config = {
    "filters": {
        "carbon_source": ["D-glucose"]
    },
    "BrentLab/harbison_2004": {
        "dataset": {
            "harbison_2004": {
                "carbon_source": {
                    "field": "condition",
                    "path": "media.carbon_source"
                }
            }
        }
    }
}

# Save to file
config_path = Path("/tmp/glucose_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(glucose_config, f)

print("Configuration:")
print(yaml.dump(glucose_config, default_flow_style=False))

Configuration:
BrentLab/harbison_2004:
  dataset:
    harbison_2004:
      carbon_source:
        field: condition
        path: media.carbon_source
filters:
  carbon_source:
  - D-glucose



### Mode 0: Conditions Only

Get just the matching field values without retrieving any data. This is the fastest mode for exploration.

In [5]:
# Create resolver and run filter
resolver = DatasetFilterResolver(config_path)

results = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="conditions"
)

# Display results
result = results["BrentLab/harbison_2004"]
print(f"Included: {result['included']}")

if result['included']:
    print("\nMatching conditions:")
    for field, values in result['matching_field_values'].items():
        print(f"  {field}: {values}")

Included: True

Matching conditions:
  condition: ['YPD', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90', 'HEAT']


**Understanding the results:**

The resolver found all condition field values where `media.carbon_source` contains D-glucose:
- YPD: Rich media with glucose
- HEAT: Heat stress with glucose
- H2O2Hi/H2O2Lo: Oxidative stress with glucose
- Acid, Alpha, BUT14, BUT90, RAPA: Various stresses with glucose

Notice GAL is NOT included - it uses galactose, not glucose.

## Example 2: Multiple Filter Criteria

Filter for samples with both D-glucose AND 30°C temperature.

In [6]:
multi_filter_config = {
    "filters": {
        "carbon_source": ["D-glucose"],
        "temperature_celsius": [30]
    },
    "BrentLab/harbison_2004": {
        "dataset": {
            "harbison_2004": {
                "carbon_source": {
                    "field": "condition",
                    "path": "media.carbon_source"
                },
                "temperature_celsius": {
                    "field": "condition",
                    "path": "temperature_celsius"
                }
            }
        }
    }
}

config_path = Path("/tmp/multi_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(multi_filter_config, f)

resolver = DatasetFilterResolver(config_path)
results = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="conditions"
)

matching = results["BrentLab/harbison_2004"]["matching_field_values"]["condition"]
print("Conditions with D-glucose AND 30°C:")
print(matching)

Conditions with D-glucose AND 30°C:
['YPD', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90']


Notice HEAT is now excluded - it has glucose but uses a temperature shift to 37°C, not 30°C.

**Multiple filter criteria use AND logic** - all criteria must match.

## Example 3: Repo-Level Filtering

Filter Kemmeren 2014, which has repo-level experimental conditions (no field specification needed).

In [7]:
repo_level_config = {
    "filters": {
        "carbon_source": ["D-glucose"],
        "temperature_celsius": [30]
    },
    "BrentLab/kemmeren_2014": {
        "dataset": {
            "kemmeren_2014": {
                "carbon_source": {
                    "path": "media.carbon_source"  # No 'field' - repo-level property
                },
                "temperature_celsius": {
                    "path": "temperature_celsius"
                }
            }
        }
    }
}

config_path = Path("/tmp/repo_level_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(repo_level_config, f)

resolver = DatasetFilterResolver(config_path)
results = resolver.resolve_filters(
    repos=[("BrentLab/kemmeren_2014", "kemmeren_2014")],
    mode="conditions"
)

result = results["BrentLab/kemmeren_2014"]
print(f"Included: {result['included']}")

if result['included']:
    if result['matching_field_values']:
        print("\nMatching field values:")
        for field, values in result['matching_field_values'].items():
            print(f"  {field}: {values}")
    else:
        print("\nNo field-level filtering - entire dataset matches at repo level")

Included: True

No field-level filtering - entire dataset matches at repo level


**Key difference**: Repo-level filtering includes/excludes the entire dataset. Since kemmeren_2014 has D-glucose at 30°C at the repo level, ALL samples in the dataset match.

## Example 4: Multiple Datasets

Filter across both harbison_2004 (field-level) and kemmeren_2014 (repo-level) in a single query.

In [8]:
multi_dataset_config = {
    "filters": {
        "carbon_source": ["D-glucose"]
    },
    "BrentLab/harbison_2004": {
        "dataset": {
            "harbison_2004": {
                "carbon_source": {
                    "field": "condition",
                    "path": "media.carbon_source"
                }
            }
        }
    },
    "BrentLab/kemmeren_2014": {
        "dataset": {
            "kemmeren_2014": {
                "carbon_source": {
                    "path": "media.carbon_source"
                }
            }
        }
    }
}

config_path = Path("/tmp/multi_dataset_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(multi_dataset_config, f)

resolver = DatasetFilterResolver(config_path)
results = resolver.resolve_filters(
    repos=[
        ("BrentLab/harbison_2004", "harbison_2004"),
        ("BrentLab/kemmeren_2014", "kemmeren_2014")
    ],
    mode="conditions"
)

print("Results across multiple datasets:\n")
for repo_id, result in results.items():
    print(f"{repo_id}:")
    print(f"  Included: {result['included']}")
    if result['included'] and result.get('matching_field_values'):
        for field, values in result['matching_field_values'].items():
            print(f"  {field}: {values[:3]}...")  # Show first 3
    print()

Results across multiple datasets:

BrentLab/harbison_2004:
  Included: True
  condition: ['YPD', 'RAPA', 'H2O2Hi']...

BrentLab/kemmeren_2014:
  Included: True



## Example 5: Retrieving Sample Metadata (Mode 1)

Once you've identified matching conditions, retrieve sample-level metadata.

In [9]:
# Reuse the glucose filter config
glucose_config = {
    "filters": {
        "carbon_source": ["D-glucose"]
    },
    "BrentLab/harbison_2004": {
        "dataset": {
            "harbison_2004": {
                "carbon_source": {
                    "field": "condition",
                    "path": "media.carbon_source"
                }
            }
        }
    }
}

config_path = Path("/tmp/glucose_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(glucose_config, f)

resolver = DatasetFilterResolver(config_path)

# Use samples mode
results = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="samples"
)

df_samples = results["BrentLab/harbison_2004"]["data"]

print(f"Retrieved {len(df_samples)} samples")
print(f"Columns: {list(df_samples.columns)}")
print("\nFirst few rows:")
df_samples.head()

Retrieved 310 samples
Columns: ['sample_id', 'db_id', 'regulator_locus_tag', 'regulator_symbol', 'condition', 'target_locus_tag', 'target_symbol', 'effect', 'pvalue']

First few rows:


,sample_id,db_id,regulator_locus_tag,regulator_symbol,condition,target_locus_tag,target_symbol,effect,pvalue
0,1,0.0,YSC0017,MATA1,YPD,YBL068W,PRS4,0.859943,0.728345
1,2,1.0,YAL051W,OAF1,YPD,YAL015C,NTG1,1.307908,0.176895
2,3,2.0,YBL005W,PDR3,YPD,YAL005C,SSA1,0.765846,0.923576
3,4,3.0,YBL008W,HIR1,YPD,YAL030W,SNC1,0.925666,0.697461
4,5,4.0,YBL021C,HAP3,YPD,YAR035W,YAT1,0.837194,0.857071


In [10]:
# Analyze condition distribution
print("Condition distribution in retrieved samples:")
print(df_samples['condition'].value_counts())

Condition distribution in retrieved samples:
condition
YPD       204
H2O2Hi     39
H2O2Lo     28
RAPA       14
BUT14       8
HEAT        6
Alpha       5
BUT90       4
Acid        2
Name: count, dtype: int64


## Example 6: Retrieving Full Data (Mode 2)

Retrieve all measurements for matching samples (many rows per sample).

In [11]:
# Use full_data mode
results_full = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="full_data"
)

df_full = results_full["BrentLab/harbison_2004"]["data"]

print(f"Retrieved {len(df_full):,} rows (measurements)")
print(f"DataFrame shape: {df_full.shape}")
print("\nFirst few rows:")
df_full.head()

Retrieved 1,930,060 rows (measurements)
DataFrame shape: (1930060, 9)

First few rows:


,sample_id,db_id,regulator_locus_tag,regulator_symbol,condition,target_locus_tag,target_symbol,effect,pvalue
0,1,0.0,YSC0017,MATA1,YPD,YAL001C,TFC3,1.697754,0.068705
1,1,0.0,YSC0017,MATA1,YPD,YAL002W,VPS8,NaN,NaN
2,1,0.0,YSC0017,MATA1,YPD,YAL003W,EFB1,NaN,NaN
3,1,0.0,YSC0017,MATA1,YPD,YAL004W,YAL004W,0.745342,0.835929
4,1,0.0,YSC0017,MATA1,YPD,YAL005C,SSA1,NaN,NaN


In [12]:
# Compare modes
n_samples = len(df_samples)
n_measurements = len(df_full)
measurements_per_sample = n_measurements / n_samples if n_samples > 0 else 0

print(f"Samples mode: {n_samples} samples")
print(f"Full data mode: {n_measurements:,} measurements")
print(f"Average measurements per sample: {measurements_per_sample:,.0f}")

Samples mode: 310 samples
Full data mode: 1,930,060 measurements
Average measurements per sample: 6,226


## Example 7: Using Repo-Wide Properties

For repositories where a property applies to ALL datasets, use repo-wide configuration.

In [13]:
repo_wide_config = {
    "filters": {
        "temperature_celsius": [30]
    },
    "BrentLab/kemmeren_2014": {
        "temperature_celsius": {  # Repo-wide property (no dataset key)
            "path": "temperature_celsius"
        },
        "dataset": {
            "kemmeren_2014": {
                "carbon_source": {  # Dataset-specific property
                    "path": "media.carbon_source"
                }
            }
        }
    }
}

print("Configuration with repo-wide property:")
print(yaml.dump(repo_wide_config, default_flow_style=False))

Configuration with repo-wide property:
BrentLab/kemmeren_2014:
  dataset:
    kemmeren_2014:
      carbon_source:
        path: media.carbon_source
  temperature_celsius:
    path: temperature_celsius
filters:
  temperature_celsius:
  - 30



In this example:
- `temperature_celsius` is defined at repo level - applies to ALL datasets in BrentLab/kemmeren_2014
- `carbon_source` is defined at dataset level - applies only to the kemmeren_2014 dataset

This is useful when a repository has multiple datasets that share some common properties.

## Creating a Reusable Configuration File

For production use, save your configuration to a YAML file.

In [14]:
# Create a comprehensive filter config
comprehensive_config = {
    "filters": {
        "carbon_source": ["D-glucose", "D-galactose"],
        "temperature_celsius": [30]
    },
    "BrentLab/harbison_2004": {
        "dataset": {
            "harbison_2004": {
                "carbon_source": {
                    "field": "condition",
                    "path": "media.carbon_source"
                },
                "temperature_celsius": {
                    "field": "condition",
                    "path": "temperature_celsius"
                }
            }
        }
    },
    "BrentLab/kemmeren_2014": {
        "dataset": {
            "kemmeren_2014": {
                "carbon_source": {
                    "path": "media.carbon_source"
                },
                "temperature_celsius": {
                    "path": "temperature_celsius"
                }
            }
        }
    }
}

# Save to file
config_file = Path("my_comprehensive_filter.yaml")
with open(config_file, 'w') as f:
    yaml.dump(comprehensive_config, f)

print(f"Saved configuration to {config_file}")
print("\nYou can now use this config file in your analysis:")
print(f"  resolver = DatasetFilterResolver('{config_file}')")

Saved configuration to my_comprehensive_filter.yaml

You can now use this config file in your analysis:
  resolver = DatasetFilterResolver('my_comprehensive_filter.yaml')


## Summary

### Key Concepts

1. **Configuration Structure**:
   - `filters`: What values you want
   - Repository mappings: Where to find properties in each dataset

2. **Three Property Types**:
   - **Repo-wide**: Applies to all datasets (paths get `experimental_conditions.` prepended)
   - **Dataset-specific**: Applies to one dataset (paths get `experimental_conditions.` prepended)
   - **Field-level**: Per-sample variation (requires `field`, no prepending)

3. **Three Output Modes**:
   - `conditions`: Fast exploration, no data download
   - `samples`: Sample metadata, one row per sample
   - `full_data`: All measurements, many rows per sample

### Best Practices

1. **Start with `conditions` mode** to verify your filters work
2. **Use `field` parameter** when properties vary between samples
3. **Omit `field` parameter** when properties are at repo/config level
4. **Keep paths simple** - use minimal nesting that works
5. **Version control your configs** - they document your filter criteria

### Common Patterns

**Field-level filtering** (per-sample variation):
```yaml
BrentLab/harbison_2004:
  dataset:
    harbison_2004:
      carbon_source:
        field: condition
        path: media.carbon_source
```

**Repo-level filtering** (all samples share conditions):
```yaml
BrentLab/kemmeren_2014:
  dataset:
    kemmeren_2014:
      carbon_source:
        path: media.carbon_source
```

**Repo-wide property** (applies to all datasets in repo):
```yaml
BrentLab/kemmeren_2014:
  temperature_celsius:
    path: temperature_celsius
```

## Next Steps

- See [my_filter_config.yaml](my_filter_config.yaml) for a working example
- Explore the [experimental_conditions_tutorial.ipynb](experimental_conditions_tutorial.ipynb) to understand datacard structures
- Read the [DatasetFilterResolver API documentation](../../api/datainfo/filter_resolver.md) for detailed reference
- Try filtering across additional datasets in the BrentLab collection